In [21]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q http://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install -q findspark

In [22]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"

In [23]:
import findspark
findspark.init()

In [24]:
from pyspark.sql import SparkSession
from pyspark import SparkContext, SparkConf
from typing import NamedTuple
from datetime import datetime
import numpy as np
from pyspark.sql.functions import udf, col
from pyspark.sql import functions as func
from pyspark.sql.types import DoubleType

## Решение задач для данных велопарковок Сан-Франциско

# **Инициализация сессии**

In [25]:
spark = SparkSession.builder\
        .master("local[*]")\
        .appName("InteractiveBikeSession_L1")\
        .getOrCreate()

# **Загрузка данных**

In [26]:
trip_data = spark.read\
.option("header", True)\
.option("inferSchema", True)\
.option("timestampFormat", 'M/d/y H:m')\
.csv("trip.csv")

print("Trips")
trip_data.printSchema()

station_data = spark.read\
.option("header", True)\
.option("inferSchema", True)\
.option("timestampFormat", 'M/d/y H:m')\
.csv("station.csv")

print("Stations")
station_data.printSchema()

Trips
root
 |-- id: integer (nullable = true)
 |-- duration: integer (nullable = true)
 |-- start_date: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: integer (nullable = true)
 |-- end_date: timestamp (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: integer (nullable = true)
 |-- bike_id: integer (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- zip_code: string (nullable = true)

Stations
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- dock_count: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- installation_date: string (nullable = true)



1. Найти велосипед с максимальным временем пробега.

In [27]:
# Вычисляем суммарную продолжительность поездок для каждого велосипеда и находим рекордсмена
bike_max_duration = (
    trip_data
    .groupBy("bike_id")
    .agg({"duration": "sum"})
    .withColumnRenamed("sum(duration)", "total_duration")
    .sort("total_duration", ascending=False)
    .first()
)

print(f"Велосипед под номером {bike_max_duration['bike_id']} использовался дольше всех ({bike_max_duration['total_duration']} минут).")


Велосипед под номером 535 использовался дольше всех (18611693 минут).


2.Найти наибольшее геодезическое расстояние между станциями.

In [28]:
# Установка библиотеки для расчёта расстояний
!pip install haversine

# Импорт необходимой функции
from haversine import haversine

# Определяем функцию для расчёта расстояния между двумя координатами
def compute_distance(lat_a, lon_a, lat_b, lon_b):
    return haversine((lat_a, lon_a), (lat_b, lon_b))

# Регистрируем функцию как UDF для использования в Spark
distance_udf = func.udf(compute_distance, DoubleType())

# Извлекаем координаты станций
stations = station_data.select("id", "lat", "long")

# Создаём декартово произведение — все пары станций
all_station_combinations = stations.crossJoin(stations) \
    .toDF("id1", "lat1", "lon1", "id2", "lat2", "lon2")

# Исключаем пары, где станции совпадают
valid_pairs = all_station_combinations.filter("id1 != id2")

# Добавляем столбец с расстоянием между каждой парой
pairs_with_distance = valid_pairs.withColumn(
    "geo_distance_km",
    distance_udf("lat1", "lon1", "lat2", "lon2")
)

# Получаем пару с наибольшим расстоянием
max_distance_row = pairs_with_distance.orderBy("geo_distance_km", ascending=False).first()

# Выводим результат
print(f"Максимальное расстояние ({max_distance_row['geo_distance_km']} км) \
между станциями с ID {max_distance_row['id1']} и {max_distance_row['id2']}.")


Максимальное расстояние (69.92097253310907 км) между станциями с ID 16 и 60.


3. Найти путь велосипеда с максимальным временем пробега через станции.

In [29]:
# Находим запись с максимальной продолжительностью поездки
longest_trip = (
    trip_data
    .select("start_station_name", "end_station_name", "duration")
    .orderBy(col("duration").desc())
    .first()
)

# Извлекаем нужные данные
origin = longest_trip["start_station_name"]
destination = longest_trip["end_station_name"]
duration_secs = longest_trip["duration"]

# Вывод информации о самой длинной поездке
print(f"Поездка с наибольшей продолжительностью ({duration_secs} сек.) началась на станции \"{origin}\" и завершилась на \"{destination}\".")

Поездка с наибольшей продолжительностью (17270400 сек.) началась на станции "South Van Ness at Market" и завершилась на "2nd at Folsom".


4. Найти количество велосипедов в системе.

In [30]:
# Подсчитываем общее количество уникальных велосипедов в данных
unique_bike_ids = trip_data.select("bike_id").distinct()
total_bikes = unique_bike_ids.count()

# Вывод результата
print(f"Всего велосипедов зарегистрировано в системе: {total_bikes}")

Всего велосипедов зарегистрировано в системе: 700


5. Найти пользователей потративших на поездки более 3 часов.

In [31]:
# Регистрируем DataFrame как временную таблицу
trip_data.createOrReplaceTempView("trip_table")

# Выполняем SQL-запрос для поиска пользователей с суммарным временем поездок более 3 часов
query = """
SELECT zip_code
FROM trip_table
GROUP BY zip_code
HAVING SUM(duration) > 10800
"""

long_usage_zip_codes = spark.sql(query)

# Отображаем найденные zip-коды
long_usage_zip_codes.show()



+--------+
|zip_code|
+--------+
|   94102|
|   95134|
|   84606|
|   80305|
|   60070|
|   95519|
|   43085|
|   91910|
|   77339|
|   48063|
|   85022|
|    1090|
|    2136|
|   11722|
|   95138|
|   94610|
|   94404|
|   80301|
|   91326|
|   90742|
+--------+
only showing top 20 rows

